first test validate_sequence:
- sequence shorter than k
- sequence contain character that not "A","T","C","G"
- negative k, or k is not integer

second test update_kmer_count:
- new k-mer first adding to the dictionart
- same k-mer seconde time with same next character
- same k-mer with different next character

third test count_kmers_with_context:
- test with short, repeat, sample sequence, to see whether we get expected results
- NOTE: we don't test k is larger than length of sequence here, as it was tested in the first funtion

forth test write_results_to_file:
- create a simple kmer data, to see whether we get expected output file
- whether the result listed in alphabetical order


In [139]:
import kmer_analyzer
import importlib  

importlib.reload(kmer_analyzer)

from kmer_analyzer import validate_sequence, update_kmer_count,count_kmers_with_context,write_results_to_file


In [118]:
# test sequence shorter than k
def test_validate_sequence_shorter_than_k():
    sequence = "ACG"
    k = 5
    result = validate_sequence(sequence, k)
    if result == False:
        print("Yeah! Test passed!✅ Sequence is shorter than k.")
        
    assert result == False

test_validate_sequence_shorter_than_k()

Yeah! Test passed!✅ Sequence is shorter than k.


In [119]:
# test sequence with invalid characters
def test_validate_sequence_invalid_characters():
    sequence = ">ACGSTX"
    k = 3
    result = validate_sequence(sequence, k)
    if result == False:
        print("Yeah! Test passed!✅: Sequence contains invalid characters.")
        
    assert result == False

test_validate_sequence_invalid_characters()

Yeah! Test passed!✅: Sequence contains invalid characters.


In [120]:
# test negative k value
def test_validate_sequence_negative_k():
    sequence = "ATCGGTAC"
    k = -1
    result = validate_sequence(sequence, k)
    if result == False:
        print("Yeah! Test passed!✅: Negative k value is invalid.")
    assert result == False

test_validate_sequence_negative_k()   

Yeah! Test passed!✅: Negative k value is invalid.


In [121]:
def test_update_kmer_count_new_kmer():
    kmer_data = {} # new kmer, here we start with an empty dictionary
    kmer = "ATC"
    next_char = "G"
    
    updated_kmer_data = update_kmer_count(kmer_data, kmer, next_char)
    
    assert updated_kmer_data[kmer]['count'] == 1
    assert updated_kmer_data[kmer]['next_chars'] == {"G": 1}
    print("Yeah! Test passed!✅: New k-mer is initialized and counted correctly.")
test_update_kmer_count_new_kmer()


Yeah! Test passed!✅: New k-mer is initialized and counted correctly.


In [122]:
def test_update_kmer_count_same_kmer_same_next_char():
    kmer_data = {"ATC": {'count': 1, 'next_chars': {"G": 1}}}
    kmer = "ATC"
    next_char = "G"     
    updated_kmer_data = update_kmer_count(kmer_data, kmer, next_char)
    assert updated_kmer_data[kmer]['count'] == 2
    assert updated_kmer_data[kmer]['next_chars'] == {"G": 2}

In [123]:
def test_update_kmer_count_same_kmer_different_next_char():
    kmer_data = {"ATC": {'count': 2, 'next_chars': {"G": 2}}}
    kmer = "ATC"
    next_char = "A"     
    updated_kmer_data = update_kmer_count(kmer_data, kmer, next_char)
    assert updated_kmer_data[kmer]['count'] == 3
    assert updated_kmer_data[kmer]['next_chars'][next_char] == 1
    assert updated_kmer_data[kmer]['next_chars'] == {"G": 2, "A": 1}

In [ ]:
def test_count_kmers_simple():
    sequence = "ATGATCCATC" # repeat AT three times， with two different next characters.
    k = 2

    result = count_kmers_with_context(sequence, k)
    
    # maunally write the expected results for this sequence
    assert "AT" in result
    assert "TG" in result
    assert "GA" in result
    assert "TC" in result
    assert "CC" in result
    assert "CA" in result

    assert result["AT"] == {'count': 3, 'next_chars': {'G': 1, 'C': 2}}
    assert result["TG"] == {'count': 1, 'next_chars': {'A': 1}}
    assert result["GA"] == {'count': 1, 'next_chars': {'T': 1}}
    assert result["TC"] == {'count': 1, 'next_chars': {'C': 1}}
    assert result["CC"] == {'count': 1, 'next_chars': {'A': 1}}
    assert result["CA"] == {'count': 1, 'next_chars': {'T': 1}}
    
test_count_kmers_simple()

In [145]:
# test_kmer_analyzer.py
#from kmer_analyzer import write_results_to_file
import tempfile
from pathlib import Path

def test_write_results_to_file_basic():
    # make a simple kmer_data dict to test the output format
    kmer_data = {
        "AT": {"count": 3, "next_chars": {"G": 1, "C": 2}},
        "TG": {"count": 1, "next_chars": {"A": 1}},
    }

    # use a temporary directory to avoid creating actual files on disk
    with tempfile.TemporaryDirectory() as tmpdir:
        out_path = Path(tmpdir) / "out.txt"

        write_results_to_file(kmer_data, out_path)

        with open(out_path, "r") as f:
            lines = f.read().splitlines()

    assert len(lines) == 2

    # expected output format:
    #    - AT: C:2 G:1
    #    - CG: A:2
    assert lines[0] == "AT C:2 G:1"
    assert lines[1] == "TG A:1"
test_write_results_to_file_basic()

In [ ]:
import sys
import tempfile
from pathlib import Path

from kmer_analyzer import main

def test_main_integration(tmp_path):
    # create a temporary input file with some sequences
    input_path = tmp_path / "seqs.txt" # define the input file path
    input_path.write_text("ATGC\nAT3X\nATGATC\n")  # some line with invalid character, will skip.
    output_path = tmp_path / "out.txt" # define the output file path

    # input arguments for main(): input file, k, output file
    sys.argv = ["kmer_analyzer.py", str(input_path), "2", str(output_path)]

    main()

    assert output_path.exists()
    content = output_path.read_text().strip().splitlines()
    assert any(line.startswith("AT") for line in content)

    test_main_integration()
